## Add Dependencies

In [24]:
import os
import getpass
from pathlib import Path
from typing import List, Optional, Dict, Any, Tuple
from dataclasses import dataclass
from datetime import datetime, timedelta
import json, re, uuid, numpy as np

from pypdf import PdfReader
from dateutil import tz
import dateparser
from pydantic import BaseModel, Field, ValidationError
from icalendar import Calendar, Event as ICS
from tqdm import tqdm

from langchain.tools import tool
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate

LOCAL_TZ = tz.gettz(os.getenv("TZ", "America/Los_Angeles"))

# ---------- Event schema ----------
class ExtractedEvent(BaseModel):
    id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    title: str
    start: datetime
    end: Optional[datetime] = None
    all_day: bool = False
    location: Optional[str] = None
    grades: Optional[List[str]] = None
    audience: Optional[List[str]] = None
    actions: Optional[List[str]] = None
    deadline: Optional[datetime] = None
    source_pdf: str
    source_span: Optional[str] = None
    confidence: float = 0.0
    notes: Optional[str] = None

class ExtractionBundle(BaseModel):
    events: List[ExtractedEvent]
    warnings: Optional[List[str]] = None

# ---------- PDF → text ----------
def pdf_to_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    return "\n".join([p.extract_text() or "" for p in reader.pages]).strip()

DATE_PAT = re.compile(r"(?:\b[A-Za-z]{3,9}\b \d{1,2}, \d{4})")
def guess_newsletter_date(text: str, default: datetime) -> datetime:
    m = DATE_PAT.search(text)
    if m:
        d = dateparser.parse(m.group(0), settings={"TIMEZONE":"UTC"})
        if d:
            return d.astimezone(LOCAL_TZ).replace(hour=9, minute=0, second=0, microsecond=0)
    return default

def _coerce_dt(x):
    if not x: return None
    d = dateparser.parse(x)
    if not d: return None
    if not d.tzinfo: d = d.replace(tzinfo=LOCAL_TZ)
    return d.astimezone(LOCAL_TZ)

# ---------- Helpers ----------
def dedupe_events(events: List[ExtractedEvent]) -> List[ExtractedEvent]:
    seen: Dict[tuple, ExtractedEvent] = {}
    for ev in events:
        k = (ev.title.strip().lower(),
             ev.start.date().isoformat() if ev.start else "none",
             (ev.location or "").strip().lower())
        prev = seen.get(k)
        if not prev or ev.confidence > prev.confidence:
            seen[k] = ev
    return list(seen.values())

def list_upcoming(events: List[ExtractedEvent], days: int = 14, grade: Optional[str] = None):
    now = datetime.now(tz=LOCAL_TZ); until = now + timedelta(days=days)
    out = []
    for e in events:
        if not e.start: continue
        if now <= e.start <= until and (not grade or (e.grades and grade in e.grades)):
            out.append(e)
    return out

def list_deadlines(events: List[ExtractedEvent], days:int=14):
    now = datetime.now(tz=LOCAL_TZ); until = now + timedelta(days=days)
    return [e for e in events if e.deadline and now <= e.deadline <= until]

def to_ics(events: List[ExtractedEvent], calendar_name="School Events") -> bytes:
    cal = Calendar(); cal.add("prodid","-//ParentHelper//School Calendar//EN"); cal.add("version","2.0")
    cal.add("X-WR-CALNAME", calendar_name)
    for e in events:
        ve = ICS(); ve.add("uid", e.id); ve.add("summary", e.title)
        if e.all_day:
            ve.add("dtstart", e.start.date())
            if e.end: ve.add("dtend", e.end.date())
        else:
            ve.add("dtstart", e.start); ve.add("dtend", e.end or (e.start + timedelta(hours=1)))
        if e.location: ve.add("location", e.location)
        desc = []
        if e.actions:  desc.append("Actions: " + ", ".join(e.actions))
        if e.grades:   desc.append("Grades: " + ", ".join(e.grades))
        if e.audience: desc.append("Audience: " + ", ".join(e.audience))
        if e.deadline: desc.append("Deadline: " + e.deadline.isoformat())
        desc.append("Source: " + e.source_pdf)
        if e.notes:    desc.append("Notes: " + e.notes)
        ve.add("description", "\n".join(desc)); cal.add_component(ve)
    return cal.to_ical()


## LLM adapter + extraction prompt

In [25]:
from openai import OpenAI
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")
client = OpenAI()  # requires OPENAI_API_KEY to be set

EXTRACTION_SYSTEM = """You extract school-related events from newsletter text.
Resolve relative dates against the provided Newsletter-Date (local timezone).
Output ONLY valid JSON per the schema. Lower confidence if unsure and include the source sentence in notes."""
SCHEMA_HINT = """
Return JSON:
{"events":[
  {"title":str,"start":ISO-8601 tz,"end":ISO-8601 tz|null,"all_day":bool,
   "location":str|null,"grades":[str]|null,"audience":[str]|null,
   "actions":[str]|null,"deadline":ISO-8601 tz|null,"source_pdf":str,
   "source_span":str|null,"confidence":float,"notes":str|null}],
 "warnings":[str]|null}
"""

def build_user_prompt(text: str, newsletter_date: datetime, source_pdf: str) -> str:
    return f"""Newsletter-Date: {newsletter_date.isoformat()}
Timezone: America/Los_Angeles
Rules:
- If only a date, set all_day=true and omit time.
- If time window present, set start/end and all_day=false.
- Include RSVP/payment deadlines and “what to bring/wear”.

SOURCE FILE: {source_pdf}
TEXT:
\"\"\"{text[:12000]}\"\"\""""

def llm_json(system: str, user: str, schema_hint: str) -> str:
    prompt = f"{user}\n\nSchema:\n{schema_hint}\n\nReturn ONLY JSON."
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"system","content":system},{"role":"user","content":prompt}],
        temperature=0.1,
    )
    return resp.choices[0].message.content.strip()

def extract_events_from_pdf(pdf_path: Path) -> ExtractionBundle:
    text = pdf_to_text(pdf_path)
    default_dt = datetime.fromtimestamp(pdf_path.stat().st_mtime, tz=LOCAL_TZ)
    nl_date = guess_newsletter_date(text, default_dt)

    raw_json = llm_json(EXTRACTION_SYSTEM, build_user_prompt(text, nl_date, pdf_path.name), SCHEMA_HINT)
    payload = json.loads(raw_json)

    events = []
    for ev in payload.get("events", []):
        ev["start"]    = _coerce_dt(ev.get("start"))
        ev["end"]      = _coerce_dt(ev.get("end"))
        ev["deadline"] = _coerce_dt(ev.get("deadline"))
        ev["source_pdf"] = ev.get("source_pdf") or pdf_path.name
        try:
            events.append(ExtractedEvent(**ev))
        except ValidationError as ve:
            print("[skip] invalid event:", ve)
    return ExtractionBundle(events=events, warnings=payload.get("warnings"))

def extract_from_folder(pdf_dir: Path) -> List[ExtractedEvent]:
    all_events: List[ExtractedEvent] = []
    for pdf in tqdm(sorted(pdf_dir.glob("*.pdf"))):
        try:
            bundle = extract_events_from_pdf(pdf)
            all_events.extend(bundle.events)
        except Exception as e:
            print(f"[WARN] {pdf.name}: {e}")
    return sorted(dedupe_events(all_events), key=lambda e: e.start)


## RAG indexing

In [26]:
# ====================================================
# RAG STORAGE: in-memory Qdrant + session manifest
# ====================================================

# !pip install qdrant-client openai numpy

from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue
import numpy as np, hashlib
from pathlib import Path

# --- in-memory Qdrant ---
client_q = QdrantClient(location=":memory:")   # ephemeral, per notebook session
COLLECTION = "scout_newsletters"

def ensure_collection(dim: int = 1536):
    existing = [c.name for c in client_q.get_collections().collections]
    if COLLECTION not in existing:
        client_q.recreate_collection(
            collection_name=COLLECTION,
            vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
        )

# --- chunker ---
def split_into_chunks(txt: str, max_chars=800, overlap=100):
    chunks, i = [], 0
    while i < len(txt):
        j = min(len(txt), i + max_chars)
        chunk = txt[i:j]
        k = chunk.rfind(". ")
        if k > 300:
            j = i + k + 2
            chunk = txt[i:j]
        chunk = chunk.strip()
        if chunk:
            chunks.append(chunk)
        i = max(j - overlap, j)
    return chunks

# --- helpers for stable IDs + fingerprints ---
def file_fingerprint(path: Path) -> dict:
    st = path.stat()
    return {"mtime": int(st.st_mtime), "size": int(st.st_size)}

def point_id_for(source_pdf: str, chunk_idx: int) -> int:
    h = hashlib.blake2b(f"{source_pdf}:{chunk_idx}".encode(), digest_size=8).hexdigest()
    return int(h, 16)

# --- embeddings (reuse your OpenAI client) ---
from openai import OpenAI
client = OpenAI()
EMB_MODEL = "text-embedding-3-small"

def embed_texts(texts: list[str]) -> np.ndarray:
    if not texts:
        return np.zeros((0, 1536), dtype="float32")
    resp = client.embeddings.create(model=EMB_MODEL, input=texts)
    return np.array([d.embedding for d in resp.data], dtype="float32")

# --- session manifest (to skip re-indexing) ---
MANIFEST_CACHE: dict[str, dict] = {}   # { "file.pdf": {"mtime":..., "size":...} }

def delete_chunks_for(source_pdf: str):
    client_q.delete(
        collection_name=COLLECTION,
        points_selector=Filter(
            must=[FieldCondition(key="source_pdf", match=MatchValue(value=source_pdf))]
        )
    )

def index_folder_incremental(pdf_folder: str, max_chars=800, overlap=100) -> dict:
    ensure_collection()
    pdf_dir = Path(pdf_folder)
    files = sorted(pdf_dir.glob("*.pdf"))

    indexed, skipped = 0, 0
    for path in files:
        src = path.name
        fp = file_fingerprint(path)
        if MANIFEST_CACHE.get(src) == fp:
            skipped += 1
            continue  # unchanged this session

        delete_chunks_for(src)
        text = pdf_to_text(path)
        chunks = split_into_chunks(text, max_chars=max_chars, overlap=overlap)
        vecs = embed_texts(chunks)

        points = []
        for i, (chunk, vec) in enumerate(zip(chunks, vecs)):
            points.append(PointStruct(
                id=point_id_for(src, i),
                vector=vec.tolist(),
                payload={
                    "source_pdf": src,
                    "chunk_index": i,
                    "text": chunk,
                    "fingerprint": fp
                }
            ))
        if points:
            for j in range(0, len(points), 256):
                client_q.upsert(collection_name=COLLECTION, points=points[j:j+256])

        MANIFEST_CACHE[src] = fp
        indexed += 1

    return {"indexed": indexed, "skipped": skipped, "total_files": len(files)}

# --- Qdrant search wrapper ---
def qdrant_search(query: str, k: int = 5, filter_sources: list[str] | None = None):
    qvec = embed_texts([query])[0].tolist()
    q_filter = None
    if filter_sources:
        q_filter = Filter(must=[FieldCondition(key="source_pdf", match=MatchValue(value=filter_sources))])
    res = client_q.search(
        collection_name=COLLECTION,
        query_vector=qvec,
        limit=k,
        query_filter=q_filter,
        with_payload=True,
    )
    out = []
    for r in res:
        payload = r.payload or {}
        out.append(({"source_pdf": payload.get("source_pdf"), "text": payload.get("text","")}, float(r.score)))
    return out

# --- plug into agent hooks ---
def build_rag_index_from_folder(pdf_folder: str):
    return index_folder_incremental(pdf_folder)

def rag_search(query: str, k=5):
    return qdrant_search(query, k=k)


In [27]:
# ====================================================
# 🔍 Qdrant Index Health Check (Improved)
# ====================================================

from collections import Counter

def index_health_check(collection: str = COLLECTION, sample_query: str = "Spirit Day"):
    """Print stats of the in-memory Qdrant index and show a sample search."""
    try:
        cols = client_q.get_collections().collections
    except Exception as e:
        print("❌ Could not reach Qdrant client:", e)
        return

    if not cols:
        print("❌ No collections in Qdrant client.")
        return

    col_names = [c.name for c in cols]
    if collection not in col_names:
        print(f"⚠️  Collection '{collection}' not found. Available: {col_names}")
        return

    # Show basic stats
    info = client_q.get_collection(collection)
    total_points = getattr(info, "points_count", None) or getattr(info, "vectors_count", 0)
    print(f"🗂️  Collection: {collection} — {total_points:,} vectors")

    # Fetch all payloads (safe even if empty)
    try:
        scroll_result, _ = client_q.scroll(collection_name=collection, with_payload=True, limit=10000)
    except Exception as e:
        print("⚠️  Scroll failed:", e)
        scroll_result = []

    payloads = [p.payload for p in scroll_result if p.payload]
    if not payloads:
        print("⚠️  No chunks indexed yet.")
        return

    # Count by file
    by_file = Counter(p.get("source_pdf", "unknown") for p in payloads)
    print("\n📄 Chunks per file:")
    for f, c in by_file.most_common():
        print(f"  {f:<40} {c:>4}")

    # Optional sample query
    if sample_query:
        print(f"\n🎯 Sample search: '{sample_query}'")
        try:
            results = qdrant_search(sample_query, k=3)
            if not results:
                print("   (no matches returned)")
            for i, (chunk, score) in enumerate(results, 1):
                snippet = chunk["text"].replace("\n", " ")[:180]
                print(f"\nTop {i}: {chunk['source_pdf']} (score={score:.3f})\n  {snippet}...")
        except Exception as e:
            print("⚠️  Sample search failed:", e)


## Agent: tools + planner + single ask() helper

In [28]:
# -------- Agent state & tool registry --------
AGENT_STATE = {"all_events": [], "pdf_folder": None}
TOOLS: Dict[str, Any] = {}
def tool(name):
    def deco(fn): TOOLS[name]=fn; return fn
    return deco

@tool("read_pdfs")
def read_pdfs(folder: str) -> Dict[str, Any]:
    # List PDFs in the folder
    pdf_dir = Path(folder)
    out = []
    for pdf in sorted(pdf_dir.glob("*.pdf")):
        out.append(pdf.name)
    return {"pdfs": out, "count": len(out)}

@tool("extract_events")
def extract_events(folder: str) -> Dict[str, Any]:
    events = extract_from_folder(Path(folder))
    AGENT_STATE["all_events"] = events
    return {"events": len(events), "sample_titles": [e.title for e in events[:5]]}

@tool("dedupe")
def dedupe_tool(_: str = "") -> Dict[str, Any]:
    ev = dedupe_events(AGENT_STATE["all_events"]); AGENT_STATE["all_events"] = ev
    return {"events": len(ev)}


@tool("build_index")
def build_index_tool(folder: str) -> dict:
    return build_rag_index_from_folder(folder)

# ---- Query parsing ----
PARSE_SYSTEM = """You parse a parent's question about school events.
Output ONLY JSON:
{"intent":"date_query"|"event_query"|"instructions_query",
 "date": "YYYY-MM-DD" or null,
 "keywords":[str]}
- date_query: "what's on Oct 21"
- event_query: "when is Spirit Day"
- instructions_query: "any instructions for Spirit Day"
Assume America/Los_Angeles; resolve dates to YYYY-MM-DD."""

def parse_query(q: str) -> dict:
    msg = [{"role":"system","content":PARSE_SYSTEM},
           {"role":"user","content":q + "\nReturn ONLY JSON."}]
    resp = client.chat.completions.create(model="gpt-4o-mini", temperature=0, messages=msg)
    return json.loads(resp.choices[0].message.content.strip())

@tool("parse_query")
def parse_query_tool(query: str) -> dict:
    return parse_query(query)

# ---- Answer composition ----
ANSWER_SYSTEM = """You answer questions about school events using:
(1) structured events and (2) retrieved newsletter snippets.
- Be concise and exact for dates.
- For instructions, list short bullets and cite the source filename in parentheses.
- If unknown, say so."""

def find_events_by_date(events, iso_date: str):
    from datetime import date
    target = date.fromisoformat(iso_date)
    return [e for e in events if e.start and e.start.date() == target]

def find_events_by_keyword(events, keywords: List[str]):
    kws = [k.lower() for k in keywords]
    hits = []
    for e in events:
        hay = " ".join([e.title, e.location or "", e.notes or "", " ".join(e.actions or [])]).lower()
        if all(k in hay for k in kws):
            hits.append(e)
    if not hits:
        for e in events:
            hay = " ".join([e.title, e.location or "", e.notes or "", " ".join(e.actions or [])]).lower()
            if any(k in hay for k in kws):
                hits.append(e)
    return hits

@tool("answer_query")
def answer_query_tool(query: str) -> dict:
    intent = parse_query(query)
    intent_type = intent.get("intent")
    date_iso = intent.get("date")
    keywords = intent.get("keywords") or []
    events = AGENT_STATE["all_events"]

    selected: List[ExtractedEvent] = []
    if intent_type == "date_query" and date_iso:
        selected = find_events_by_date(events, date_iso)
        search_text = f"{date_iso} " + " ".join(keywords)
    elif intent_type in ("event_query","instructions_query") and keywords:
        selected = find_events_by_keyword(events, keywords)
        search_text = " ".join(keywords)
    else:
        search_text = query

    snippets = rag_search(search_text, k=5)

    context = {
        "query": query,
        "intent": intent,
        "events": [{
            "title": e.title,
            "start": e.start.isoformat() if e.start else None,
            "all_day": e.all_day,
            "location": e.location,
            "grades": e.grades,
            "actions": e.actions,
            "deadline": e.deadline.isoformat() if e.deadline else None,
            "source": e.source_pdf,
            "notes": e.notes,
            "confidence": e.confidence,
        } for e in selected],
        "snippets": [{
            "source": ch["source_pdf"],
            "similarity": sim,
            "text": ch["text"][:500]
        } for ch, sim in snippets]
    }

    prompt = f"""Answer the user's question using ONLY the context.
If giving instructions, list bullets and cite the source filename in parentheses.

CONTEXT:
{json.dumps(context, ensure_ascii=False)}
"""
    resp = client.chat.completions.create(
        model="gpt-4o-mini", temperature=0.2,
        messages=[{"role":"system","content":ANSWER_SYSTEM},
                  {"role":"user","content":prompt}]
    )
    answer = resp.choices[0].message.content.strip()
    return {"parsed_intent": intent,
            "events_matched": len(selected),
            "sources_consulted": list({s["source"] for s in context["snippets"]})[:3],
            "answer": answer}

# -------- Minimal planner (single-call convenience) --------
def ensure_prepared(pdf_folder: str):
    """Run once per session or when files change."""
    # Check if already prepared for this folder
    if (AGENT_STATE.get("pdf_folder") == pdf_folder and 
        AGENT_STATE.get("all_events") and 
        len(AGENT_STATE["all_events"]) > 0):
        print("✅ Data already prepared for this folder")
        return
    
    print("🔄 Preparing data...")
    read_pdfs(pdf_folder)
    extract_events(pdf_folder)
    dedupe_tool()
    build_index_tool(pdf_folder)
    AGENT_STATE["pdf_folder"] = pdf_folder
    print("✅ Data preparation complete")

def ask(question: str, pdf_folder: str) -> str:
    """Agent-based question answering."""
    # Ensure data is prepared (keep your existing logic)
    if (AGENT_STATE.get("pdf_folder") != pdf_folder or 
        not AGENT_STATE.get("all_events") or 
        len(AGENT_STATE["all_events"]) == 0):
        ensure_prepared(pdf_folder)
    
    # Use agent to answer
    result = agent_executor.invoke({"input": question})
    return result["output"]


In [29]:
# Create LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Create tools manually (not using @tool decorator)
from langchain.tools import BaseTool
from typing import Type

class SearchEventsTool(BaseTool):
    name: str = "search_events"
    description: str = "Search structured events by date or keywords. Use this for finding specific events."
    
    def _run(self, query: str) -> str:
        events = AGENT_STATE["all_events"]
        
        # Try to parse as date first
        try:
            from datetime import date
            target_date = date.fromisoformat(query)
            matching_events = [e for e in events if e.start and e.start.date() == target_date]
            if matching_events:
                result = f"Found {len(matching_events)} events on {query}:\n"
                for e in matching_events:
                    result += f"- {e.title} at {e.start.strftime('%I:%M %p') if e.start else 'All day'}"
                    if e.location:
                        result += f" ({e.location})"
                    result += "\n"
                return result
        except:
            pass
        
        # Search by keywords
        keywords = query.lower().split()
        matching_events = []
        for e in events:
            hay = " ".join([e.title, e.location or "", e.notes or "", " ".join(e.actions or [])]).lower()
            if any(k in hay for k in keywords):
                matching_events.append(e)
        
        if matching_events:
            result = f"Found {len(matching_events)} events matching '{query}':\n"
            for e in matching_events:
                result += f"- {e.title} on {e.start.strftime('%Y-%m-%d') if e.start else 'TBD'}"
                if e.location:
                    result += f" at {e.location}"
                result += "\n"
            return result
        else:
            return f"No events found matching '{query}'"

class SearchDocumentsTool(BaseTool):
    name: str = "search_documents"
    description: str = "Search raw document chunks using RAG. Use this for finding detailed information."
    
    def _run(self, query: str) -> str:
        try:
            snippets = rag_search(query, k=5)
            if not snippets:
                return f"No relevant information found for '{query}'"
            
            result = f"Found {len(snippets)} relevant document snippets for '{query}':\n\n"
            for i, (chunk, score) in enumerate(snippets, 1):
                result += f"{i}. From {chunk['source_pdf']} (relevance: {score:.3f}):\n"
                result += f"   {chunk['text'][:200]}...\n\n"
            return result
        except Exception as e:
            return f"Error searching documents: {str(e)}"

# Create tools list
tools = [SearchEventsTool(), SearchDocumentsTool()]

# Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful school events assistant. You can search for events and documents to answer questions about school activities, dates, and instructions.
    
    Available tools:
    - search_events: Find events by date or keywords
    - search_documents: Search raw document content for detailed information
    
    Always use the appropriate tool to find information before answering. Be helpful and provide specific details when available."""),
    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# Create agent
agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [30]:
PDF_DIR = "data/newsletters"  # put your 7–8 PDFs here

# Prepare (run when PDFs change)
ensure_prepared(PDF_DIR)

# Check Qdrant is properly loaded
index_health_check()

# Now ask questions (like a chatbot, but from the notebook)
print(ask("Are there any events on October 21st I need to plan for?", PDF_DIR))
print(ask("When is Unity Day?", PDF_DIR))
print(ask("Are there any instructions for Unity Day?", PDF_DIR))

🔄 Preparing data...


 25%|██▌       | 1/4 [00:25<01:15, 25.20s/it]

[WARN] Notes from Oak _ Smore Newsletters _ Oct_3_2025.pdf: Expecting value: line 1 column 1 (char 0)


 50%|█████     | 2/4 [00:43<00:42, 21.16s/it]

[WARN] Notes from Oak _ Smore Newsletters_Oct_10_2025.pdf: Expecting value: line 1 column 1 (char 0)


...

100%|██████████| 4/4 [01:35<00:00, 23.83s/it]

[WARN] Notes from Oak _ Smore Newsletters_Sept_26_2025.pdf: Expecting value: line 1 column 1 (char 0)
✅ Data preparation complete


> Entering new AgentExecutor chain...



Invoking: `search_documents` with `{'query': 'Unity Day instructions'}`


Found 5 relevant document snippets for 'Unity Day instructions':

1. From Notes from Oak _ Smore Newsletters_Oct_10_2025.pdf (relevance: 0.386):
   o make the
WAT a wonderful success!
WAT Team Planners & Helpers
Madhu Prasad
Sherrill Marquardt
Amy Leung
Kristi Schramm
Raluca Oanta
Sandy Chen
Steph Shum
Dan Rowan
Alba Garza
Alina Tan
Belle Nguyen
...

2. From Notes from Oak _ Smore Newsletters_Sept_26_2025.pdf (relevance: 0.381):
   rdinary education into an LASD
Education! Learn more and donate atLASDEducationPartners.org.
6th Grade Demonstrating
the 6 C's
While Working on Early Settlement Building
Projects
Join LASD's AI Playla...

3. From Notes from Oak _ Smore Newsletters_Sept_26_2025.pdf (relevance: 0.329):
   for a shift or 2 (pro tip: sign up early to get your ideal duty and shift!).
1.1K
10/14/25, 10:10 PM Notes from Oak | Smore Newsletters
https://secure.smore.com/n/1tb0m-notes-from-oak?ref=email 5/17
.

/var/folders/dq/7_w0fl7j3s3fs7t6pbr2jtmh0000gq/T/ipykernel_34669/2420366429.py:116: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  res = client_q.search(



Invoking: `search_events` with `{'query': 'Unity Day'}`


No events found matching 'Unity Day'I couldn't find specific instructions for Unity Day in the available documents or events. It might be helpful to check with your school administration or the event organizers for the most accurate and detailed information. If you have any other questions or need assistance with something else, feel free to ask!

> Finished chain.
I couldn't find specific instructions for Unity Day in the available documents or events. It might be helpful to check with your school administration or the event organizers for the most accurate and detailed information. If you have any other questions or need assistance with something else, feel free to ask!


In [31]:
# Test event extraction directly
from app.tools import AGENT_STATE
from app.data_processing import pdf_to_text
from pathlib import Path

# Check what events were actually extracted
print(f"Number of events extracted: {len(AGENT_STATE['all_events'])}")

# Test PDF text extraction
pdf_path = Path("data/newsletters/Notes from Oak _ Smore Newsletters_Oct_10_2025.pdf")
text = pdf_to_text(pdf_path)
print(f"PDF text length: {len(text)}")
print(f"First 500 chars: {text[:500]}")

# Check if text extraction is working
if len(text) < 100:
    print("❌ PDF text extraction is failing - PDFs might be image-based")
else:
    print("✅ PDF text extraction is working")

Number of events extracted: 0
PDF text length: 17637
First 500 chars: 488Share Translate Accessibility
10/14/25, 10:06 PM Notes from Oak | Smore Newsletters
https://secure.smore.com/n/fe2cp-notes-from-oak?ref=email 1/16
Professional Development Day at Oak
While our Cougars have the day off today, the learning continues for
our staff! Our teachers will spend time learning together—sharing
insights from the morning and reflecting on our compelling
question:
“How can I design learning experiences that help students develop
the dispositions that matter most in our
✅ PDF text extraction is working


In [32]:
# Test LLM response directly
from pathlib import Path
import json
from openai import OpenAI
from app.data_processing import pdf_to_text, guess_newsletter_date
from app.tools import EXTRACTION_SYSTEM, SCHEMA_HINT, build_user_prompt
from datetime import datetime
from dateutil import tz

# Define the client
client = OpenAI()
LOCAL_TZ = tz.gettz("America/Los_Angeles")

# Test on one PDF
pdf_path = Path("data/newsletters/Notes from Oak _ Smore Newsletters_Oct_10_2025.pdf")

try:
    print("Testing LLM response...")
    
    # Extract text
    text = pdf_to_text(pdf_path)
    print(f"Text length: {len(text)}")
    
    # Get newsletter date
    default_dt = datetime.fromtimestamp(pdf_path.stat().st_mtime, tz=LOCAL_TZ)
    nl_date = guess_newsletter_date(text, default_dt)
    print(f"Newsletter date: {nl_date}")
    
    # Build prompt
    user_prompt = build_user_prompt(text, nl_date, pdf_path.name)
    print(f"Prompt length: {len(user_prompt)}")
    
    # Call LLM directly
    print("Calling LLM...")
    prompt = f"{user_prompt}\n\nSchema:\n{SCHEMA_HINT}\n\nReturn ONLY JSON."
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"system","content":EXTRACTION_SYSTEM},{"role":"user","content":prompt}],
        temperature=0.1,
    )
    raw_json = resp.choices[0].message.content.strip()
    print(f"Raw LLM response length: {len(raw_json)}")
    
    # Strip markdown code blocks if present
    if raw_json.startswith("```json"):
        raw_json = raw_json[7:]  # Remove ```json
    if raw_json.endswith("```"):
        raw_json = raw_json[:-3]  # Remove ```
    raw_json = raw_json.strip()
    
    print(f"Cleaned JSON: {raw_json[:200]}...")
    
    # Try to parse JSON
    payload = json.loads(raw_json)
    print(f"✅ Successfully parsed JSON!")
    print(f"Number of events: {len(payload.get('events', []))}")
    
    for i, event in enumerate(payload.get('events', [])[:3]):
        print(f"Event {i+1}: {event.get('title')} on {event.get('start')}")
    
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

Testing LLM response...
Text length: 17637
Newsletter date: 2025-10-14 22:07:51.068897-07:00
Prompt length: 12332
Calling LLM...
Raw LLM response length: 3415
Cleaned JSON: {
  "events": [
    {
      "title": "Unity Day",
      "start": "2025-10-22T00:00:00-07:00",
      "end": null,
      "all_day": true,
      "location": null,
      "grades": null,
      "audience": ...
✅ Successfully parsed JSON!
Number of events: 7
Event 1: Unity Day on 2025-10-22T00:00:00-07:00
Event 2: Oak Halloween Parade on 2025-10-31T09:00:00-07:00
Event 3: Oak Spooktacular on 2025-10-31T14:30:00-07:00


In [33]:
# Test the fixed event extraction
from app.tools import ensure_prepared, AGENT_STATE
from app.agent import ask

# Test the complete flow
PDF_DIR = "data/newsletters"

print("🔄 Testing event extraction...")
ensure_prepared(PDF_DIR)

print(f"\n📊 Events extracted: {len(AGENT_STATE['all_events'])}")
for i, event in enumerate(AGENT_STATE['all_events'][:5]):
    print(f"  {i+1}. {event.title} on {event.start.strftime('%Y-%m-%d') if event.start else 'TBD'}")

print(f"\n🤖 Testing agent response...")
response = ask("When is Unity Day?", PDF_DIR)
print(f"Agent response: {response}")

🔄 Testing event extraction...
🔄 Preparing data...


 25%|██▌       | 1/4 [00:00<00:02,  1.06it/s]

[WARN] Notes from Oak _ Smore Newsletters _ Oct_3_2025.pdf: name 'client' is not defined


 50%|█████     | 2/4 [00:02<00:02,  1.30s/it]

[WARN] Notes from Oak _ Smore Newsletters_Oct_10_2025.pdf: name 'client' is not defined


...

 75%|███████▌  | 3/4 [00:03<00:01,  1.10s/it]

[WARN] Notes from Oak _ Smore Newsletters_Sept_19_2025.pdf: name 'client' is not defined


100%|██████████| 4/4 [00:04<00:00,  1.11s/it]

[WARN] Notes from Oak _ Smore Newsletters_Sept_26_2025.pdf: name 'client' is not defined
✅ Data preparation complete


> Entering new AgentExecutor chain...



Invoking: `search_events` with `{'query': 'Unity Day'}`


No events found matching 'Unity Day'
Invoking: `search_documents` with `{'query': 'Unity Day'}`


Found 5 relevant document snippets for 'Unity Day':

1. From Notes from Oak _ Smore Newsletters_Oct_10_2025.pdf (relevance: 0.449):
   o make the
WAT a wonderful success!
WAT Team Planners & Helpers
Madhu Prasad
Sherrill Marquardt
Amy Leung
Kristi Schramm
Raluca Oanta
Sandy Chen
Steph Shum
Dan Rowan
Alba Garza
Alina Tan
Belle Nguyen
...

2. From Notes from Oak _ Smore Newsletters_Sept_26_2025.pdf (relevance: 0.367):
   rdinary education into an LASD
Education! Learn more and donate atLASDEducationPartners.org.
6th Grade Demonstrating
the 6 C's
While Working on Early Settlement Building
Projects
Join LASD's AI Playla...

3. From Notes from Oak _ Smore Newsletters_Oct_10_2025.pdf (relevance: 0.348):
   488Share Translate Accessibility
10/14/25, 10:06 PM Notes from Oak | Smore Newsletters
https://secure.smore.com/n/fe2cp-notes-fro

In [34]:
# Create a working version of extract_events_from_pdf
from app.data_processing import ExtractedEvent, ExtractionBundle, pdf_to_text, guess_newsletter_date, _coerce_dt
from pathlib import Path
import json
from datetime import datetime
from dateutil import tz
from pydantic import ValidationError

def working_extract_events_from_pdf(pdf_path: Path) -> ExtractionBundle:
    client = OpenAI()
    
    text = pdf_to_text(pdf_path)
    default_dt = datetime.fromtimestamp(pdf_path.stat().st_mtime, tz=tz.gettz("America/Los_Angeles"))
    nl_date = guess_newsletter_date(text, default_dt)

    # Call LLM directly
    prompt = f"{build_user_prompt(text, nl_date, pdf_path.name)}\n\nSchema:\n{SCHEMA_HINT}\n\nReturn ONLY JSON."
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"system","content":EXTRACTION_SYSTEM},{"role":"user","content":prompt}],
        temperature=0.1,
    )
    raw_json = resp.choices[0].message.content.strip()
    
    # Strip markdown code blocks if present
    if raw_json.startswith("```json"):
        raw_json = raw_json[7:]
    if raw_json.endswith("```"):
        raw_json = raw_json[:-3]
    raw_json = raw_json.strip()
    
    payload = json.loads(raw_json)

    events = []
    for ev in payload.get("events", []):
        ev["start"]    = _coerce_dt(ev.get("start"))
        ev["end"]      = _coerce_dt(ev.get("end"))
        ev["deadline"] = _coerce_dt(ev.get("deadline"))
        ev["source_pdf"] = ev.get("source_pdf") or pdf_path.name
        try:
            events.append(ExtractedEvent(**ev))
        except ValidationError as ve:
            print("[skip] invalid event:", ve)
    return ExtractionBundle(events=events, warnings=payload.get("warnings"))

# Test the working version
print("🧪 Testing working extract_events_from_pdf...")
bundle = working_extract_events_from_pdf(pdf_path)
print(f"✅ Extracted {len(bundle.events)} events")

# Look for Unity Day
unity_day_events = [e for e in bundle.events if "unity" in e.title.lower()]
if unity_day_events:
    event = unity_day_events[0]
    print(f"🎯 Found Unity Day: {event.title} on {event.start}")
    print(f"   Actions: {event.actions}")
    print(f"   Notes: {event.notes}")
else:
    print("❌ Unity Day not found")

🧪 Testing working extract_events_from_pdf...
✅ Extracted 7 events
🎯 Found Unity Day: Unity Day on 2025-10-22 00:00:00-07:00
   Actions: ['Wear orange']
   Notes: Unity Day is the signature event of National Bullying Prevention Month.


In [ ]:
def extract_events_from_pdf(pdf_path: Path) -> ExtractionBundle:
    from openai import OpenAI
    client = OpenAI()
    
    text = pdf_to_text(pdf_path)
    default_dt = datetime.fromtimestamp(pdf_path.stat().st_mtime, tz=tz.gettz(LOCAL_TZ))
    nl_date = guess_newsletter_date(text, default_dt)

    # Call LLM directly
    prompt = f"{build_user_prompt(text, nl_date, pdf_path.name)}\n\nSchema:\n{SCHEMA_HINT}\n\nReturn ONLY JSON."
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"system","content":EXTRACTION_SYSTEM},{"role":"user","content":prompt}],
        temperature=0.1,
    )
    raw_json = resp.choices[0].message.content.strip()
    
    # Strip markdown code blocks if present
    if raw_json.startswith("```json"):
        raw_json = raw_json[7:]
    if raw_json.endswith("```"):
        raw_json = raw_json[:-3]
    raw_json = raw_json.strip()
    
    payload = json.loads(raw_json)

    events = []
    for ev in payload.get("events", []):
        ev["start"]    = _coerce_dt(ev.get("start"))
        ev["end"]      = _coerce_dt(ev.get("end"))
        ev["deadline"] = _coerce_dt(ev.get("deadline"))
        ev["source_pdf"] = ev.get("source_pdf") or pdf_path.name
        try:
            events.append(ExtractedEvent(**ev))
        except ValidationError as ve:
            print("[skip] invalid event:", ve)
    return ExtractionBundle(events=events, warnings=payload.get("warnings"))

In [35]:
# Test the fixed event extraction
from app.tools import ensure_prepared, AGENT_STATE
from app.agent import ask

# Test the complete flow
PDF_DIR = "data/newsletters"

print("🔄 Testing event extraction...")
ensure_prepared(PDF_DIR)

print(f"\n📊 Events extracted: {len(AGENT_STATE['all_events'])}")
for i, event in enumerate(AGENT_STATE['all_events'][:5]):
    print(f"  {i+1}. {event.title} on {event.start.strftime('%Y-%m-%d') if event.start else 'TBD'}")

print(f"\n🤖 Testing agent response...")
response = ask("When is Unity Day?", PDF_DIR)
print(f"Agent response: {response}")

🔄 Testing event extraction...
🔄 Preparing data...


 25%|██▌       | 1/4 [00:00<00:02,  1.10it/s]

[WARN] Notes from Oak _ Smore Newsletters _ Oct_3_2025.pdf: name 'client' is not defined


 50%|█████     | 2/4 [00:02<00:02,  1.22s/it]

[WARN] Notes from Oak _ Smore Newsletters_Oct_10_2025.pdf: name 'client' is not defined


...

 75%|███████▌  | 3/4 [00:03<00:01,  1.23s/it]

[WARN] Notes from Oak _ Smore Newsletters_Sept_19_2025.pdf: name 'client' is not defined


100%|██████████| 4/4 [00:04<00:00,  1.22s/it]

[WARN] Notes from Oak _ Smore Newsletters_Sept_26_2025.pdf: name 'client' is not defined
✅ Data preparation complete


> Entering new AgentExecutor chain...



Invoking: `search_events` with `{'query': 'Unity Day'}`


No events found matching 'Unity Day'
Invoking: `search_documents` with `{'query': 'Unity Day'}`


Found 5 relevant document snippets for 'Unity Day':

1. From Notes from Oak _ Smore Newsletters_Oct_10_2025.pdf (relevance: 0.449):
   o make the
WAT a wonderful success!
WAT Team Planners & Helpers
Madhu Prasad
Sherrill Marquardt
Amy Leung
Kristi Schramm
Raluca Oanta
Sandy Chen
Steph Shum
Dan Rowan
Alba Garza
Alina Tan
Belle Nguyen
...

2. From Notes from Oak _ Smore Newsletters_Sept_26_2025.pdf (relevance: 0.367):
   rdinary education into an LASD
Education! Learn more and donate atLASDEducationPartners.org.
6th Grade Demonstrating
the 6 C's
While Working on Early Settlement Building
Projects
Join LASD's AI Playla...

3. From Notes from Oak _ Smore Newsletters_Oct_10_2025.pdf (relevance: 0.348):
   488Share Translate Accessibility
10/14/25, 10:06 PM Notes from Oak | Smore Newsletters
https://secure.smore.com/n/fe2cp-notes-fro